# Prediction Intervals And Calibration

P10/P90 intervals are useful only if coverage is checked against held-out actuals.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, Markdown, display
except Exception:
    def display(value):
        print(value)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image(filename={self.filename!r})"

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
PLOTS = OUT / "plots"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

def load_csv(name, **kwargs):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return pd.read_csv(path, **kwargs)

def load_json(name):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return json.loads(path.read_text())

def show_plot(name):
    path = PLOTS / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Plot not generated: `{path}`"))

In [2]:
calibration = load_csv('interval_calibration.csv')
calibration

,model,rows,coverage,avg_interval_width,p10,p90,target_coverage,calibration_gap
0,lightgbm_global_rm_pm,1728,0.695023,3248.417056,860.786658,4109.203713,0.8,-0.104977


In [3]:
selected = load_csv("selected_model_backtest_rows.csv")
selected[["model", "material_code", "month", "actual", "prediction", "residual", "abs_error"]].head(30)

,model,material_code,month,actual,prediction,residual,abs_error
0,lightgbm_global_rm_pm,100012,2025-07-01,150.0,104.133322,-45.866678,45.866678
1,lightgbm_global_rm_pm,100012,2025-08-01,103.0,106.167760,3.167760,3.167760
2,lightgbm_global_rm_pm,100012,2025-09-01,196.0,56.617960,-139.382040,139.382040
3,lightgbm_global_rm_pm,100012,2025-10-01,117.0,79.368471,-37.631529,37.631529
4,lightgbm_global_rm_pm,100012,2025-11-01,112.0,105.403201,-6.596799,6.596799
5,lightgbm_global_rm_pm,100012,2025-12-01,112.0,106.049047,-5.950953,5.950953
6,lightgbm_global_rm_pm,101208,2025-07-01,0.0,0.019292,0.019292,0.019292
7,lightgbm_global_rm_pm,101208,2025-08-01,0.0,0.036983,0.036983,0.036983
8,lightgbm_global_rm_pm,101208,2025-09-01,0.0,0.037729,0.037729,0.037729
9,lightgbm_global_rm_pm,101208,2025-10-01,0.0,0.036616,0.036616,0.036616


In [4]:
forward = load_csv("forecast_results_v7.csv", parse_dates=["forecast_period"])
forward.groupby("horizon").agg(
    materials=("material_id", "nunique"),
    p10_sum=("forecast_p10", "sum"),
    p50_sum=("forecast_p50", "sum"),
    p90_sum=("forecast_p90", "sum"),
)

,materials,p10_sum,p50_sum,p90_sum
horizon,,,,
1,288,234197.24,477233.86,1118005.25
2,288,231353.53,471439.20,1104430.40
3,288,220292.22,448899.13,1051626.22
4,288,211114.76,430197.59,1007814.51
5,288,240249.77,489567.27,1146898.75
6,288,200219.04,407994.84,955800.67
7,288,194021.40,395365.77,926214.72
8,288,200305.24,408170.39,956211.97
9,288,186565.44,380172.37,890621.56
